In [1]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/ulrikthygepedersen/online-retail-dataset/online_retail.csv', encoding='latin1') # common encoding for this dataset

# 1. Look for non-standard StockCodes (non-physical items)
# Standard stock codes are usually 5 digits, sometimes with a letter at the end.
# Let's find codes that are very short or purely alphabetical.
non_standard_codes = df[df['StockCode'].str.contains('^[A-Za-z]+$', regex=True, na=False)]['StockCode'].unique()
print("Non-standard StockCodes (purely alphabetical):", non_standard_codes)

# Let's check descriptions for these non-standard codes
for code in non_standard_codes:
    desc = df[df['StockCode'] == code]['Description'].unique()
    print(f"Code: {code}, Descriptions: {desc}")

# 2. Check Entity Identification (StockCode vs Description)
# How many StockCodes have multiple Descriptions?
stockcode_desc_counts = df.groupby('StockCode')['Description'].nunique()
codes_with_multiple_descs = stockcode_desc_counts[stockcode_desc_counts > 1]
print(f"\nNumber of StockCodes with >1 Description: {len(codes_with_multiple_descs)}")
if len(codes_with_multiple_descs) > 0:
    print("Example StockCodes with multiple descriptions:")
    example_code = codes_with_multiple_descs.index[0]
    print(f"Code {example_code}: {df[df['StockCode'] == example_code]['Description'].unique()}")

# How many Descriptions have multiple StockCodes?
desc_stockcode_counts = df.groupby('Description')['StockCode'].nunique()
descs_with_multiple_codes = desc_stockcode_counts[desc_stockcode_counts > 1]
print(f"\nNumber of Descriptions with >1 StockCode: {len(descs_with_multiple_codes)}")

Non-standard StockCodes (purely alphabetical): ['POST' 'D' 'DOT' 'M' 'S' 'AMAZONFEE' 'm' 'DCGSSBOY' 'DCGSSGIRL' 'PADS'
 'B' 'CRUK']
Code: POST, Descriptions: ['POSTAGE' nan]
Code: D, Descriptions: ['Discount']
Code: DOT, Descriptions: ['DOTCOM POSTAGE' nan]
Code: M, Descriptions: ['Manual']
Code: S, Descriptions: ['SAMPLES']
Code: AMAZONFEE, Descriptions: ['AMAZON FEE']
Code: m, Descriptions: ['Manual']
Code: DCGSSBOY, Descriptions: ['BOYS PARTY BAG']
Code: DCGSSGIRL, Descriptions: ['GIRLS PARTY BAG']
Code: PADS, Descriptions: ['PADS TO MATCH ALL CUSHIONS']
Code: B, Descriptions: ['Adjust bad debt']
Code: CRUK, Descriptions: ['CRUK Commission']

Number of StockCodes with >1 Description: 650
Example StockCodes with multiple descriptions:
Code 10080: ['GROOVY CACTUS INFLATABLE' nan 'check']

Number of Descriptions with >1 StockCode: 172


In [2]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules

# 1. Tải dữ liệu
file_path = "/kaggle/input/datasets/ulrikthygepedersen/online-retail-dataset/online_retail.csv"
df = pd.read_csv(file_path, encoding='latin1')

# --- THỰC HIỆN DATA CLEANSING (DM_C2) ---

# Loại bỏ các khoảng trắng thừa và in hoa toàn bộ Description để đồng nhất
df['Description'] = df['Description'].str.strip().str.upper()

# Xóa các dòng không có mã hóa đơn
df.dropna(axis=0, subset=['InvoiceNo'], inplace=True)

# Loại bỏ các hóa đơn bị hủy (bắt đầu bằng 'C')
df['InvoiceNo'] = df['InvoiceNo'].astype('str')
df = df[~df['InvoiceNo'].str.contains('C')]

# Loại bỏ các giao dịch có Quantity <= 0 (outliers/noise)
df = df[df['Quantity'] > 0]

# Tinh chỉnh: Loại bỏ các mã StockCode phi vật lý (M, D, POST, S, AMAZONFEE...)
# Thường mã sản phẩm thật sẽ chứa chữ số, các mã chỉ toàn chữ cái là nhiễu
df = df[~df['StockCode'].str.contains('^[a-zA-Z]+$', regex=True)]

# --- ENTITY IDENTIFICATION: Tạo từ điển ánh xạ StockCode -> Description ---
# Lấy Description đầu tiên (không bị null) cho mỗi StockCode để hiển thị sau này
item_name_mapping = df.dropna(subset=['Description']).groupby('StockCode')['Description'].first().to_dict()

# --- DATA TRANSFORMATION & AGGREGATION ---

# Gom nhóm theo StockCode thay vì Description để đếm cho chính xác
basket = (df[df['Country'] == "United Kingdom"]
          .groupby(['InvoiceNo', 'StockCode'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

# Rời rạc hóa dữ liệu (Discretization)
def encode_units(x):
    return True if x >= 1 else False

basket_sets = basket.map(encode_units) # Dùng .map() thay cho .applymap() trên pandas mới

# --- TÌM TẬP PHỔ BIẾN & LUẬT KẾT HỢP ---
frequent_itemsets = fpgrowth(basket_sets, min_support=0.03, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# --- POST-PROCESSING: Phục hồi tên sản phẩm để dễ đọc ---
# Đổi frozenset StockCode sang tên Description
rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join([item_name_mapping.get(item, item) for item in x]))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join([item_name_mapping.get(item, item) for item in x]))

rules = rules.sort_values(['confidence', 'lift'], ascending=[False, False])

# --- BUSINESS-READY OUTPUT FORMATTING ---
print("Top 10 Recommendations based on Association Rules:\n" + "-"*80)
for index, row in rules.head(10).iterrows():
    antecedent = row['antecedents']
    consequent = row['consequents']
    conf_pct = round(row['confidence'] * 100, 2)
    supp_pct = round(row['support'] * 100, 2)
    lift_val = round(row['lift'], 2)
    
    print(f"IF a customer buys: [{antecedent}]")
    print(f"Then the best item to recommend is: [{consequent}]")
    print(f"Confidence: {conf_pct}% | Support: {supp_pct}% | Lift: {lift_val}x")
    print("-" * 80)

Top 10 Recommendations based on Association Rules:
--------------------------------------------------------------------------------
IF a customer buys: [PINK REGENCY TEACUP AND SAUCER]
Then the best item to recommend is: [GREEN REGENCY TEACUP AND SAUCER]
Confidence: 82.08% | Support: 3.09% | Lift: 16.42x
--------------------------------------------------------------------------------
IF a customer buys: [GREEN REGENCY TEACUP AND SAUCER]
Then the best item to recommend is: [ROSES REGENCY TEACUP AND SAUCER]
Confidence: 75.05% | Support: 3.75% | Lift: 14.65x
--------------------------------------------------------------------------------
IF a customer buys: [ROSES REGENCY TEACUP AND SAUCER]
Then the best item to recommend is: [GREEN REGENCY TEACUP AND SAUCER]
Confidence: 73.25% | Support: 3.75% | Lift: 14.65x
--------------------------------------------------------------------------------
IF a customer buys: [JUMBO BAG PINK POLKADOT]
Then the best item to recommend is: [JUMBO BAG RED RETR

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
# --- RÚT TRÍCH LUẬT KẾT HỢP (ASSOCIATION RULES) (DM_C6) ---

# Tạo ra các luật kết hợp từ frequent itemsets
# Rút trích các luật có độ tin cậy (confidence) tối thiểu 50% trước
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

# Sau đó mới giữ lại các luật có tương quan dương (lift > 1)
rules = rules[rules['lift'] > 1.0]

# --- POST-PROCESSING: Phục hồi tên sản phẩm để dễ đọc ---
# Đổi frozenset StockCode sang tên Description
rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join([item_name_mapping.get(item, item) for item in x]))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join([item_name_mapping.get(item, item) for item in x]))

# Sắp xếp như bạn đã làm
rules = rules.sort_values(['confidence', 'lift'], ascending=[False, False])

# --- BUSINESS-READY OUTPUT FORMATTING ---
print("Top 10 Recommendations based on Association Rules:\n" + "-"*80)
for index, row in rules.head(10).iterrows():
    antecedent = row['antecedents']
    consequent = row['consequents']
    conf_pct = round(row['confidence'] * 100, 2)
    supp_pct = round(row['support'] * 100, 2)
    lift_val = round(row['lift'], 2)
    
    print(f"IF a customer buys: [{antecedent}]")
    print(f"Then the best item to recommend is: [{consequent}]")
    print(f"Confidence: {conf_pct}% | Support: {supp_pct}% | Lift: {lift_val}x")
    print("-" * 80)

Top 10 Recommendations based on Association Rules:
--------------------------------------------------------------------------------
IF a customer buys: [PINK REGENCY TEACUP AND SAUCER]
Then the best item to recommend is: [GREEN REGENCY TEACUP AND SAUCER]
Confidence: 82.08% | Support: 3.09% | Lift: 16.42x
--------------------------------------------------------------------------------
IF a customer buys: [GREEN REGENCY TEACUP AND SAUCER]
Then the best item to recommend is: [ROSES REGENCY TEACUP AND SAUCER]
Confidence: 75.05% | Support: 3.75% | Lift: 14.65x
--------------------------------------------------------------------------------
IF a customer buys: [ROSES REGENCY TEACUP AND SAUCER]
Then the best item to recommend is: [GREEN REGENCY TEACUP AND SAUCER]
Confidence: 73.25% | Support: 3.75% | Lift: 14.65x
--------------------------------------------------------------------------------
IF a customer buys: [JUMBO BAG PINK POLKADOT]
Then the best item to recommend is: [JUMBO BAG RED RETR

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag